#### Trying to put satlink data into my common format. 

In [11]:
import pandas as pd
import numpy as np
import re

In [27]:
data = pd.read_parquet(r"Data\Satlink_live\September 24 2025\data\combined_positions.parquet")

In [23]:
data

,StoredTime,Type,Latitude,Longitude,Speed,source_file
0,2025-09-24 06:57:20 UTC,DR,14º29.352'S,169º33.540'W,0.26 kn,SLX_404307_positions_20250924_155130.csv
1,2025-09-23 06:57:20 UTC,DR,14º34.002'S,169º29.220'W,0.06 kn,SLX_404307_positions_20250924_155130.csv
2,2025-09-22 06:58:00 UTC,DR,14º33.642'S,169º27.840'W,0.36 kn,SLX_404307_positions_20250924_155130.csv
3,2025-09-21 06:57:13 UTC,DR,14º41.904'S,169º25.020'W,0.04 kn,SLX_404307_positions_20250924_155130.csv
4,2025-09-20 06:57:04 UTC,DR,14º40.842'S,169º24.900'W,0.52 kn,SLX_404307_positions_20250924_155130.csv
...,...,...,...,...,...,...
3032,2025-08-29 13:26:14 UTC,DR,6º37.958'N,149º26.820'W,1.28 kn,ISD_511349_positions_20250924_155130.csv
3033,2025-08-28 13:26:15 UTC,DR,6º13.773'N,149º45.900'W,0.80 kn,ISD_511349_positions_20250924_155130.csv
3034,2025-08-27 13:26:20 UTC,DR,5º55.138'N,149º50.880'W,0.58 kn,ISD_511349_positions_20250924_155130.csv
3035,2025-08-26 13:26:13 UTC,DR,5º41.604'N,149º53.760'W,0.78 kn,ISD_511349_positions_20250924_155130.csv


In [9]:
def dms_to_decimal(coord_str: str) -> float:
    """
    Convert coordinate from 'DDºMM.mmmX' format to decimal degrees.
    
    Example: '14º29.352'S' -> -14.4892
    """
    # regex to capture degrees, minutes, and hemisphere
    match = re.match(r"(\d+)º([\d\.]+)'?([NSEW])", coord_str.strip())
    if not match:
        raise ValueError(f"Invalid coordinate format: {coord_str}")
    
    deg = float(match.group(1))
    minutes = float(match.group(2))
    hemi = match.group(3).upper()
    
    decimal = deg + minutes / 60.0
    
    if hemi in ["S", "W"]:
        decimal = -decimal
    
    return decimal

In [19]:

def extract_prefix(filename: str) -> str:
    """
    Extract prefix before '_positions' from a filename.
    
    Example:
    'SLX_404307_positions_20250924_155130.csv' -> 'SLX_404307'
    """
    match = re.match(r"(.+?)_positions", filename)
    if match:
        return match.group(1)
    else:
        raise ValueError(f"Filename not in expected format: {filename}")

In [28]:
data['Latitude'] = data['Latitude'].apply(dms_to_decimal)
data['Longitude'] = data['Longitude'].apply(dms_to_decimal)

In [30]:
data["StoredTime"] = pd.to_datetime(data["StoredTime"])
data["BuoyName"] = data["source_file"].apply(extract_prefix)

In [37]:
data['Timestamp'] = data['StoredTime'].dt.tz_localize(None)
data = data.sort_values("Timestamp")
data["MinOfTimes"] = data.groupby(["BuoyName"])['Timestamp'].transform("min")
data["MaxOfTimes"] = data.groupby(["BuoyName"])['Timestamp'].transform("max")
data = data.sort_values(by = ['MinOfTimes', "Timestamp"])
data = data.drop_duplicates()
cols_to_keep=['BuoyName', 'Timestamp', 'Latitude', 'Longitude']
data = data[cols_to_keep]


In [41]:
data

,BuoyName,Timestamp,Latitude,Longitude
501,SLX_481841,2025-08-18 13:30:10,20.569700,-162.495
500,SLX_481841,2025-08-19 01:30:11,20.557200,-162.443
499,SLX_481841,2025-08-19 13:30:11,20.631500,-162.392
498,SLX_481841,2025-08-20 01:30:11,20.603100,-162.299
497,SLX_481841,2025-08-20 13:30:17,20.621400,-162.274
...,...,...,...,...
620,SLX_501035,2025-09-24 03:04:35,7.324567,-163.324
619,SLX_501035,2025-09-24 07:04:39,7.373150,-163.239
618,SLX_501035,2025-09-24 11:04:34,7.409767,-163.157
617,SLX_501035,2025-09-24 15:04:40,7.448967,-163.069
